In [1]:
import yaml

with open("creditCardFraud/config/model_config.yaml", "r") as file:
    config = yaml.safe_load(file)

print(config)

{'model': {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'scale_pos_weight': 599.4761904761905, 'random_state': 42}, 'threshold': 0.2325, 'data': {'test_size': 0.2, 'random_state': 42}}


In [3]:
from xgboost import XGBClassifier

model_params = config["model"]

model = XGBClassifier(
    **model_params,
    objective = "binary:logistic",
    eval_metric = "logloss",
    n_jobs = -1
)

print(model.get_params())

{'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.8, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': 'logloss', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 6, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': 599.4761904761905, 'subsample': 0.8, 'tree_method': None, 'validate_parameters': None, 'verbosity': None}


In [6]:
import pandas as pd
from sklearn.metrics import average_precision_score,roc_auc_score

X_train = pd.read_csv("creditCardFraud/data/processed/X_train_scaled.csv")
X_test = pd.read_csv("creditCardFraud/data/processed/X_test_scaled.csv")

y_train = pd.read_csv("creditCardFraud/data/processed/y_train.csv").squeeze("columns")
y_test = pd.read_csv("creditCardFraud/data/processed/y_test.csv").squeeze("columns")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (226980, 30)
y_train: (226980,)
X_test: (56746, 30)
y_test: (56746,)


In [7]:
model.fit(X_train,y_train)

repro_proba = model.predict_proba(X_test)[:, 1]

print("PR-AUC:", average_precision_score(y_test, repro_proba))
print("ROC-AUC:", roc_auc_score(y_test, repro_proba))

PR-AUC: 0.8248384759168829
ROC-AUC: 0.9761103301934559


In [8]:
threshold = config["threshold"]

repro_pred = (repro_proba >= threshold).astype(int)

print("Threshold:", threshold)
print("Fraud predictions:", repro_pred.sum())

Threshold: 0.2325
Fraud predictions: 84
